In [3]:
import pandas as pd
import os
from IPython.display import display

# ── Assay metadata: filename stem → (AID label, Cell Line, Assay Type) ──────
AID_META = {
    "AID_1346978": ("AID_1346978", "HepG2",                "Caspase"),
    "AID_1346979": ("AID_1346979", "CHO-K1",               "Caspase Viability"),
    "AID_1346980": ("AID_1346980", "CHO-K1",               "Caspase"),
    "AID_1346981": ("AID_1346981", "HepG2",                "caspase Viability"),
    "AID_654":     ("AID_654",     "HepG2",                "Caspase"),
    "AID_655":     ("AID_655",     "Jurkat",               "Caspase"),
    "AID_656":     ("AID_656",     "HUV-EC-C",             "Caspase"),
    "AID_657":     ("AID_657",     "SH-SY5Y",              "Caspase"),
    "AID_658":     ("AID_658",     "BJ",                   "Caspase"),
    "AID_659":     ("AID_659",     "MRC-5",                "Caspase"),
    "AID_660":     ("AID_660",     "Mesangial",            "Caspase"),
    "AID_661":     ("AID_661",     "SK-N-SH",              "Caspase"),
    "AID_663":     ("AID_663",     "H-4-II-E",             "Caspase"),
    "AID_664":     ("AID_664",     "HEK293",               "Caspase"),
    "AID_665":     ("AID_665",     "N2a",                  "Caspase"),
    "AID_666":     ("AID_666",     "NIH 3T3",              "Caspase"),
    "AID_667":     ("AID_667",     "Renal Proximal Tubule","Caspase"),
    "AID_720634":  ("AID_720634",  "HepG2",                "Mitochondria Viability"),
    "AID_720635":  ("AID_720635",  "HepG2",                "Mitochondria"),
    # combined HepG2 caspase (two independent screens merged)
    "combined_hepg2_casepase": ("combined_hepg2", "HepG2", "Caspase"),
}

CLEANED_PATH = "/Users/yaminichitikela/Documents/casepase/Casepase-activation/cleaned_datasets"

# ── Build per-assay summary table ────────────────────────────────────────────
rows = []
for fname in sorted(os.listdir(CLEANED_PATH)):
    if not fname.endswith(".csv"):
        continue
    stem = fname.replace("_cleaned.csv", "").replace(".csv", "")
    if stem not in AID_META:
        continue
    aid_label, cell_line, assay_type = AID_META[stem]
    df = pd.read_csv(os.path.join(CLEANED_PATH, fname), low_memory=False)
    counts = df["PUBCHEM_ACTIVITY_OUTCOME"].value_counts()
    active       = int(counts.get("Active", 0))
    inactive     = int(counts.get("Inactive", 0))
    inconclusive = int(counts.get("Inconclusive", 0))
    total        = active + inactive + inconclusive
    activation_rate = round(active / total * 100, 2) if total > 0 else 0.0
    rows.append({
        "AID":              aid_label,
        "Cell Line":        cell_line,
        "Assay Type":       assay_type,
        "Active":           active,
        "Inactive":         inactive,
        "Inconclusive":     inconclusive,
        "Total Tested":     total,
        "Activation Rate (%)": activation_rate,
    })

summary_df = pd.DataFrame(rows)

# Sort: first by Assay Type, then by Cell Line
summary_df = summary_df.sort_values(["Assay Type", "Cell Line"]).reset_index(drop=True)

print(f"Assays loaded: {len(summary_df)}")
print(summary_df.to_string(index=False))

Assays loaded: 20
           AID             Cell Line             Assay Type  Active  Inactive  Inconclusive  Total Tested  Activation Rate (%)
       AID_658                    BJ                Caspase       6      1301            26          1333                 0.45
   AID_1346980                CHO-K1                Caspase     115      7320           236          7671                 1.50
       AID_663              H-4-II-E                Caspase      19      1268            46          1333                 1.43
       AID_664                HEK293                Caspase      17      1274            42          1333                 1.28
       AID_656              HUV-EC-C                Caspase       5      1313            15          1333                 0.38
   AID_1346978                 HepG2                Caspase     229      7164           278          7671                 2.99
       AID_654                 HepG2                Caspase      15      1296            22  

In [4]:
# ── (Optional) Detailed per-compound view — kept for reference ───────────────
# Reload and tag each file with its metadata, then produce a long-form table
# suitable for per-compound filtering.

long_dfs = []
for fname in sorted(os.listdir(CLEANED_PATH)):
    if not fname.endswith(".csv"):
        continue
    stem = fname.replace("_cleaned.csv", "").replace(".csv", "")
    if stem not in AID_META:
        continue
    aid_label, cell_line, assay_type = AID_META[stem]
    df = pd.read_csv(os.path.join(CLEANED_PATH, fname), low_memory=False,
                     usecols=["PUBCHEM_CID", "PUBCHEM_ACTIVITY_OUTCOME"])
    df = df.rename(columns={"PUBCHEM_ACTIVITY_OUTCOME": "status"})
    df["aid"]       = aid_label
    df["cell_line"] = cell_line
    df["assay_type"]= assay_type
    long_dfs.append(df)

long_df = pd.concat(long_dfs, ignore_index=True)

# Per-assay breakdown (mirrors summary_df but from long-form, for verification)
detail = (
    long_df.groupby(["aid", "cell_line", "assay_type"])["status"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)
display(detail)

status,aid,cell_line,assay_type,Active,Inactive,Inconclusive
0,AID_1346978,HepG2,Caspase,229,7164,278
1,AID_1346979,CHO-K1,Caspase Viability,1155,5457,1059
2,AID_1346980,CHO-K1,Caspase,115,7320,236
3,AID_1346981,HepG2,caspase Viability,593,6169,909
4,AID_654,HepG2,Caspase,15,1296,22
5,AID_655,Jurkat,Caspase,48,1203,82
6,AID_656,HUV-EC-C,Caspase,5,1313,15
7,AID_657,SH-SY5Y,Caspase,10,1286,37
8,AID_658,BJ,Caspase,6,1301,26
9,AID_659,MRC-5,Caspase,8,1303,23


In [5]:
# ── Export: CSV + LaTeX for paper ────────────────────────────────────────────
out_dir = "/Users/yaminichitikela/Documents/casepase/Casepase-activation/results/all_cell_line_comparisions"
os.makedirs(out_dir, exist_ok=True)

# CSV
csv_path = os.path.join(out_dir, "assay_summary_table.csv")
summary_df.to_csv(csv_path, index=False)
print(f"CSV saved → {csv_path}")

# LaTeX  (booktabs style for journals)
latex_str = summary_df.to_latex(
    index=False,
    caption="Summary of PubChem qHTS assays: compound counts and activation rates across cell lines.",
    label="tab:assay_summary",
    column_format="llcrrrrr",
    bold_rows=False,
    escape=True,
    na_rep="—",
)
# Insert \toprule / \midrule / \bottomrule via booktabs
latex_str = latex_str.replace("\\hline", "\\midrule", 1).replace("\\hline", "\\bottomrule", 1)

latex_path = os.path.join(out_dir, "assay_summary_table.tex")
with open(latex_path, "w") as f:
    f.write(latex_str)
print(f"LaTeX saved → {latex_path}")

CSV saved → /Users/yaminichitikela/Documents/casepase/Casepase-activation/results/all_cell_line_comparisions/assay_summary_table.csv
LaTeX saved → /Users/yaminichitikela/Documents/casepase/Casepase-activation/results/all_cell_line_comparisions/assay_summary_table.tex
